# LLM policy mapping

Maps policy chunks (Fabric table `policy_chunks`) to the data assets they govern, using the table / column
metadata in Neo4j. A mapper model proposes links, a second model verifies them and a third breaks ties; the code
rejects any link whose asset or quoted evidence is not real. **Metadata only: no row data is sent to any model.**

**One model is enough:** model 1 plays every role. Models 2 and 3 (verifier, tie-breaker) are optional; add them
only if model 1's token limit becomes a problem. Rate-limited calls wait and retry automatically.

* **append** (default) adds links that are missing and never touches an existing one.
* **overwrite** replaces the live links of the chosen chunks; the old ones are kept as `superseded`
  (undo: `python -m app.mapping restore --run <run_id>`).
* `WRITE = False` is a dry run: models are called, nothing is written to Neo4j.

**Before the first run**
1. Build and upload the bundle: `python scripts/build_fabric_bundle.py`, then upload `dist/lineage_mapping_bundle.zip`
   to the lakehouse at `Files/lineage_studio/`.
2. Put the secrets in Key Vault (names in the secrets cell) and set `KEY_VAULT` below.
3. Attach an Environment with `openai`, `neo4j`, `pydantic`, `pydantic-settings` (or leave the pip cell on).
4. Set this notebook's default lakehouse to the one holding `policy_chunks`.
5. Run the **smoke test** cell first: it is the only way to know this workspace can reach Azure OpenAI.

In [ ]:
# PARAMETERS. In Fabric: "..." on this cell -> Toggle parameter cell, so a pipeline can override them.
MODE = "append"            # "append" adds what is missing; "overwrite" replaces (old links kept as 'superseded')
WRITE = False              # False = dry run: nothing is written to Neo4j
CONFIRM_OVERWRITE = False  # must be True to WRITE in overwrite mode
ONLY_UNLINKED = False      # skip chunks that already have a live link (fewer model calls)
NO_AUTO_ACCEPT = False     # True = every new link is 'proposed' for review; none is auto-accepted
ACCEPT_SELF_VERIFIED = False  # one model mapping AND verifying is not independent: its links stay 'proposed'.
                              # True lets its high-confidence links be 'accepted' anyway.
DOCS = ""                  # comma-separated words of document titles to include; empty = all
LIMIT = 0                  # only the first N chunks; 0 = all
WORKERS = 4                # parallel model calls

CHUNKS_TABLE = "policy_chunks"
CHUNK_COLUMNS = {}         # map our names to the table's, e.g. {"text": "chunk_text", "chunk_id": "id"}
RESULTS_PATH = "Tables/policy_link_candidates"   # every link considered is appended here (audit / review)
KEY_VAULT = ""             # e.g. "https://my-vault.vault.azure.net/"
BUNDLE = "/lakehouse/default/Files/lineage_studio/lineage_mapping_bundle.zip"

In [ ]:
# Packages are installed from plain Python, not with the pip magic command: Fabric disables that magic in runs that
# are not interactive (a pipeline, a schedule, or the Run jobs button), which fails the whole session. Interactive runs work too.
import importlib, importlib.util, site, subprocess, sys
NEEDED = {"openai": "openai", "neo4j": "neo4j", "pydantic": "pydantic", "pydantic_settings": "pydantic-settings"}          # import name -> pip name
missing = [pip for mod, pip in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()
for mod in NEEDED:
    importlib.import_module(mod)
print("packages ready:", sorted(NEEDED.values()))

In [ ]:
import sys, zipfile, tempfile, json

assert MODE in ("append", "overwrite"), "MODE must be 'append' or 'overwrite'"
if WRITE and MODE == "overwrite" and not CONFIRM_OVERWRITE:
    raise ValueError("Overwrite replaces live links. Set CONFIRM_OVERWRITE = True to proceed.")

dest = tempfile.mkdtemp()
zipfile.ZipFile(BUNDLE).extractall(dest)
sys.path.insert(0, dest)
import app.mapping  # noqa: E402,F401  (fails here, clearly, if the bundle is missing or damaged)
print("bundle loaded from", BUNDLE)

In [ ]:
# Secrets come from Key Vault, never from a cell. Models 2 and 3 are optional: leave them unset and model 1 does every role.
try:
    from notebookutils import mssparkutils
except ImportError:
    from notebookutils import mssparkutils  # noqa: F811

def secret(name, required=True):
    try:
        return mssparkutils.credentials.getSecret(KEY_VAULT, name)
    except Exception:
        if required:
            raise
        return ""

from app.config import Settings

settings = Settings(
    mode="live",
    neo4j_uri=secret("neo4j-uri"), NEO4J_USER=secret("neo4j-user"), neo4j_password=secret("neo4j-password"),
    AZURE_OPEN_AI_KEY=secret("azure-openai-key"), AZURE_OPEN_AI_ENDPOINT=secret("azure-openai-endpoint"),
    AZURE_OPEN_AI_MODEL_NAME=secret("azure-openai-model-1"), AZURE_OPEN_AI_VERSION=secret("azure-openai-version", False) or "2024-12-01-preview",
    AZURE_OPEN_AI_MODEL_NAME_2=secret("azure-openai-model-2", False), AZURE_OPEN_AI_MODEL_NAME_3=secret("azure-openai-model-3", False),
    AZURE_OPEN_AI_KEY_2=secret("azure-openai-key-2", False), AZURE_OPEN_AI_ENDPOINT_2=secret("azure-openai-endpoint-2", False),
    AZURE_OPEN_AI_KEY_3=secret("azure-openai-key-3", False), AZURE_OPEN_AI_ENDPOINT_3=secret("azure-openai-endpoint-3", False),
)
for slot, role in ((1, "mapper"), (2, "verifier"), (3, "tie-breaker")):
    print(f"model {slot} ({role}):", settings.llm_slot(slot)["model"] or ("not set, model 1 does this too" if slot > 1 else "NOT CONFIGURED"))

In [ ]:
# SMOKE TEST: can this workspace reach each Azure OpenAI deployment? Run this first.
from app.chat.llm import AzureChatLLM

for slot in (1, 2, 3):
    if not settings.llm_slot(slot)["model"]:
        continue
    reply = AzureChatLLM(settings, slot).complete([{"role": "user", "content": "Reply with the single word: ok"}], None)
    print(f"model {slot} reachable ->", reply.content.strip()[:20])

In [ ]:
# Read the chunks from the Fabric table.
from app.mapping.store import chunks_from_rows

def read_table(name):
    """Try the usual ways to open a lakehouse Delta table (schema-enabled lakehouses differ)."""
    for loader in (lambda: spark.read.format("delta").load(f"Tables/{name}"),
                   lambda: spark.read.format("delta").load(f"Tables/dbo/{name}"),
                   lambda: spark.read.table(name)):
        try:
            return loader()
        except Exception:
            continue
    raise RuntimeError(f"Could not open table {name!r}. Check the default lakehouse and the table name.")

rows = read_table(CHUNKS_TABLE).toPandas().to_dict("records")
chunks = chunks_from_rows(rows, CHUNK_COLUMNS)
if DOCS.strip():
    needles = [d.strip().lower() for d in DOCS.split(",") if d.strip()]
    chunks = [c for c in chunks if any(n in c.doc_title.lower() or n in c.doc_id.lower() for n in needles)]
if LIMIT:
    chunks = chunks[:LIMIT]
print(f"{len(chunks)} chunks from {CHUNKS_TABLE}")

In [ ]:
# Catalogue (table / column metadata) and existing links come from Neo4j; then map.
from app.neo4j_client import Neo4jClient
from app.repository import LiveRepository
from app.mapping.catalogue import build_catalogue
from app.mapping.engine import RunConfig, run_mapping
from app.mapping.models import load_models
from app.mapping.store import Neo4jReader

client = Neo4jClient(settings)
reader = Neo4jReader(client)
catalogue = build_catalogue(LiveRepository(client).catalogue())
existing = reader.existing_links()
models = load_models(settings)
print(f"catalogue: {catalogue.tables} tables/views, {catalogue.columns} columns | existing links: {len(existing)} | models: {models.names()}")

cfg = RunConfig(mode=MODE, only_unlinked=ONLY_UNLINKED, auto_accept=not NO_AUTO_ACCEPT,
                accept_self_verified=ACCEPT_SELF_VERIFIED, workers=WORKERS)
if not models.independent_verifier:
    print("verifier is the same model as the mapper (not independent):",
          "high-confidence links will be accepted" if ACCEPT_SELF_VERIFIED else "links stay 'proposed' for review")
report = run_mapping(chunks, catalogue, models, existing, cfg,
                     lambda done, total, r: print(f"[{done}/{total}] {r.chunk.section_path[:60]}: " + (r.error or f"{len(r.links)} links")))
summary = report.summary()
print(json.dumps(summary, indent=2))

In [ ]:
# Write to Neo4j (only when WRITE = True). Append never touches existing links; overwrite supersedes, never deletes.
from app.mapping.store import MappingWriter

if WRITE:
    writer = MappingWriter(settings)
    try:
        counts = writer.apply(report.plan, models.names(), {c.chunk_id: c.doc_id for c in chunks}, catalogue.hash)
    finally:
        writer.close()
    summary["written"] = counts
    print("WRITTEN:", counts, "| undo with run id", report.run_id)
else:
    print("Dry run: nothing written to Neo4j. Set WRITE = True to apply this plan.")
client.close()

In [ ]:
# Keep every link considered (accepted, proposed, dropped) in the lakehouse, for audit and review.
import pandas as pd

result_rows = report.rows()
if result_rows:
    (spark.createDataFrame(pd.DataFrame(result_rows)).write.mode("append").format("delta").save(RESULTS_PATH))
    print(f"{len(result_rows)} rows appended to {RESULTS_PATH}")

In [ ]:
# Hand the summary to the pipeline that called this notebook.
try:
    mssparkutils.notebook.exit(json.dumps(summary))
except Exception:
    print("(not running under a pipeline)")